In [1]:
import numpy as np
from sklearn import mixture
#import torch
#import triton
#import triton.language as tl

np.random.seed(42)
#torch.manual_seed(42)

#DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [2]:
n_models = 1000
n_components = 3
n_gaussians = n_components * n_models
true_means = np.random.rand(n_gaussians, 2) * 10
true_sigmas = np.array([np.eye(2) * (0.1 + 0.9 * np.random.rand()) for _ in range(n_gaussians)])
true_weights = np.hstack([np.random.dirichlet(np.ones(n_components), size=1)[0] for _ in range(n_models)])
model_labels = np.repeat(np.arange(n_models), n_components)

n_samples = 1000
sections = []
labels = []
for i in range(len(true_weights)):
    X_i = np.random.multivariate_normal(mean=true_means[i], cov=true_sigmas[i], size=int(n_samples * true_weights[i]))
    sections.append(X_i)
    labels.append(np.full(X_i.shape[0], model_labels[i]))
X = np.vstack(sections)
labels = np.hstack(labels)

sectioned_points = []
for label in range(n_models):
    points = X[labels == label]
    sectioned_points.append(points.copy())

#X_torch = torch.from_numpy(X).float()
#X_torch_cuda = X_torch.cuda()

In [3]:
%%timeit

sklearn_gmms = []
for section in sectioned_points:
    sklearn_gmm = mixture.GaussianMixture(n_components=n_components, covariance_type='full', max_iter=1000)
    sklearn_gmm.fit(section)
    sklearn_gmms.append(sklearn_gmm)

27 s ± 3.73 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
log_probs = sklearn_gmm.score_samples(X)
sklearn_log_likelihood = np.sum(log_probs)
sklearn_log_likelihood

In [ ]:
sklearn_gmm.n_iter_

In [ ]:
from gmm import GaussianMixtureModel

In [ ]:
model = GaussianMixtureModel(n_components=n_components, max_iter=1000, verbose=True)
model.fit(X)

In [ ]:
# Find the nearest true mean for each estimated mean then compute the average distance
from scipy.spatial.distance import cdist

distances = cdist(model.mu, true_means)
min_indices = np.argmin(distances, axis=1)
closest_true_means = true_means[min_indices]
difference = model.mu - closest_true_means
average_distance = np.  mean(np.linalg.norm(difference, axis=1))

closest_true_sigma = true_sigmas[min_indices]
average_sigma_distance = np.mean(np.linalg.norm(model.sigma - closest_true_sigma, axis=(1, 2)))
average_distance, average_sigma_distance

In [ ]:
distances = cdist(sklearn_gmm.means_, true_means)
min_indices = np.argmin(distances, axis=1)
closest_true_means = true_means[min_indices]
difference = sklearn_gmm.means_ - closest_true_means
average_distance = np.mean(np.linalg.norm(difference, axis=1))

closest_true_sigma = true_sigmas[min_indices]
average_sigma_distance = np.mean(np.linalg.norm(sklearn_gmm.covariances_ - closest_true_sigma, axis=(1, 2)))
average_distance, average_sigma_distance